# Phase D / Experiment 3 — Top-k Retrieval Depth (RQ3)

Implements **MASTER_PLAN.md §10 D.3** (top-k sweep at the Exp2 winner config)
under the conventions in §13.5 (determinism) and §15.

**Frozen baseline (from the Exp1 + Exp2 winners — both `aadb0cb9`):**
- `chunk_size = 512`
- `overlap_pct = 20`
- `embedding_model = sentence-transformers/all-MiniLM-L6-v2`
- `llm = llama3.1:8b-instruct-q4_K_M`, `temperature = 0`, `seed = 42`

**Varied axis — top_k:** `{1, 3, 5, 10}` → **4 configurations**.

**Index reuse.** All 4 configs share `config_hash = aadb0cb9` (top_k is not
part of the hash inputs — see `src/chunking.py`'s `config_hash`). The index
artifact at `indices/aadb0cb9/` is loaded **once** at the top of the sweep
and reused across all 4 top_k values × 3 reps × 13 questions = 156 query
runs. `build_index` runs **zero times** in this notebook.

**Indexing row.** ONE row is appended to `results/indexing_log.csv` with the
measurement fields **copied from the existing `phase_d_exp1` row for
`aadb0cb9`** (the original Exp1 measurement; not the rebuilt
`phase_d_exp2` measurement). The fresh Exp3 `run_id` lets all 12 Exp3
query run_ids prefix-join correctly under the Phase D run-id scheme. The
`notes` column tag is `phase_d_exp3 (reused phase_d_exp1 aadb0cb9
indexing measurements)`.

**Per config:** `load_index` is structurally a no-op (the index is already
in memory); 3 reps × 13 questions = **39 query rows per top_k**.
**Exp3 total:** 1 indexing row + 156 query rows.

**Run-id scheme adapted for Exp3.** Because all 4 configs share the same
`config_hash`, a fresh-timestamp anchor per config would still produce
collisions on the rep suffix. Instead Exp3 uses ONE shared timestamp
anchor and embeds `top_k` in the query run_id:
- indexing run_id = `{exp3_ts_compact}_aadb0cb9`
- query   run_id = `{indexing_run_id}_k{k}_r{r}` for `k ∈ {1,3,5,10}`,
  `r ∈ {1,2,3}`

The standard `make_run_id` helper is used to build the indexing run_id;
the query run_id appends `_k{k}_r{r}` directly. The prefix invariant
`qry.run_id.startswith(idx.run_id)` is preserved, so Phase E joins are
unchanged. The `top_k` column in `query_log.csv` is the canonical filter
for per-config aggregation.

**Determinism.** Same MiniLM index, same chunks, same Llama+seed=42; the
only variable is `k`. Expect identical retrieved chunks for ranks ≤ k_min
across configs (e.g. the top-3 results under k=10 should be the same
chunks as the full top-3 under k=3, modulo numerical-tie ordering — FAISS
is stable). Cell 6 prints a head-of-output spot check.

**RAM pre-flight (§14 risk register; loosened §13.1 threshold).** Low risk
this experiment: no new embedding model loads, no rechunking, no rebuilds.
The pre-flight runs once at the start (model_size_mb=0) and once before
each top_k's query loop, both as defensive checks against background-
process drift. The strict 12 GB threshold from Exp1/Exp2 was sized to
catch new-allocation OOM cases like BGE-large; for reuse-only sweeps
where `model_est=0`, bloat-driven approach to 12 GB carries no swap
risk under 16 GB physical RAM and only blocks legitimate work. Cell 5's
`ram_preflight()` therefore selects between two thresholds:
`RAM_HALT_THRESHOLD_MB = 12 288` (strict; applied when model_est > 0)
and `RAM_HALT_THRESHOLD_REUSE_ONLY_MB = 13 500` (loose; applied only
when model_est == 0, with an explicit `assert` guard rejecting any
attempt to use the looser value with a non-zero model load — see §13.1).

**Partial-completion state.** This notebook supports a continuation-style
Run All: cell 2's `TOP_K_VALUES_TO_RUN` may be a subset of the full
`{1, 3, 5, 10}` Exp3 set when prior partial Run-Alls have already
persisted some configs to CSV. Cell 7's cross-run Pareto ranking always
covers the full `TOP_K_VALUES_FOR_EXP3_RANKING = (1, 3, 5, 10)` by
loading every `phase_d_exp3` indexing row from CSV (across all Run-Alls)
and aggregating their query rows by `top_k`.

**Prompt-length sanity.** Top_k=10 at chunk_size=512 → ~5120 retrieved
tokens + question + system prompt ≈ ~5300 tokens. This is well under
Llama 3.1 8B's modelfile default `num_ctx = 131 072` (128 K). The query
loop emits a loud warning if any `prompt_token_count` exceeds 7500
(defensive — should never fire); answers are not truncated by Ollama
unless `num_ctx` is exceeded, in which case the loud warning gives an
audit trail.

**CodeCarbon.** `OfflineEmissionsTracker` per (top_k, repetition) query
block: `project_name=phase_d_exp3_query_<hash>_k<k>_r{r}`. **No
`phase_d_exp3_indexing_*` block** is created (the indexing row reuses
phase_d_exp1 measurements; no real work is done at indexing time in this
notebook).

**Best-Exp3-config rule (locked, mirrors Exp1 / Exp2).** Lexicographic
Pareto: refusal_rate primary → mean `total_latency_ms` tie #1 →
`indexing_time_sec` tie #2. **Note:** `indexing_time_sec` is identical
across all 4 Exp3 configs (one shared row), so tie-break #2 is
mathematically incapable of resolving any tie. Cell 7 flags this
explicitly. If a true tie remains after tie-break #1, cell 7 prints all
tied configs and halts winner selection (the user must adjudicate).


In [ ]:
"""Cell 2 — Loop driver, frozen baseline, RAM + prompt-length thresholds.

NOTE — partial-completion state (2026-05-10).
top_k=1 was completed in the prior partial Run All on 2026-05-10 at
11:36:57Z; its rows are already in the CSVs (1 indexing row at run_id=
20260510T113657Z_aadb0cb9, notes="phase_d_exp3 (reused phase_d_exp1 ...)";
39 query rows under aadb0cb9_k1_r{1,2,3} with top_k=1, 6/13 refusals each
rep). That run was halted by the RAM pre-flight at top_k=3 with
used_now=12455 MB exceeding the 12288 threshold — heap-bloat-driven, not
new-allocation-driven (Exp3 has no new model loads). Including top_k=1
in this tuple would create duplicate run_ids under the same idx_run_id
prefix, so it is intentionally excluded. Cell 7 still ranks all 4 Exp3
configs by reading both indexing rows from CSV (the partial-run row +
this run's row).
"""

# §10 D.3 axis: retrieval depth. top_k=1 intentionally excluded — see
# the docstring above.
TOP_K_VALUES_TO_RUN: tuple[int, ...] = (3, 5, 10)

# Final Exp3 ranking set. Originally planned (1, 3, 5, 10), but top_k=10
# was dropped per §14 risk-register precedent (same pattern as the BGE-large
# drop in Exp2): RAM pre-flight halted before top_k=10 with used_now=13079
# MB exceeding the loosened 13500 threshold by 79 MB; further loosening
# would push toward swap territory on 16 GB physical RAM with no
# headroom margin. Cell 7 ranks across these 3 measured configs only.
TOP_K_VALUES_FOR_EXP3_RANKING: tuple[int, ...] = (1, 3, 5)

# Frozen from Exp1 + Exp2 winners (both aadb0cb9).
FROZEN_CHUNK_SIZE:  int = 512
FROZEN_OVERLAP_PCT: int = 20
FROZEN_EMBEDDING:   str = "sentence-transformers/all-MiniLM-L6-v2"
EXP2_WINNER_HASH:   str = "aadb0cb9"

# §14 RAM halt threshold — strict default for fresh-model-load operations
# (Exp1/Exp2 pattern). Sized to catch new-allocation OOM cases like
# BGE-large at 1024-d on top of the Llama 3.1 8B Q4 resident footprint.
RAM_HALT_THRESHOLD_MB: int = 12 * 1024  # 12 GB

# §13.1 single-machine policy — loosened threshold for REUSE-ONLY sweeps
# (no new model loads, no new allocations, just heap bloat).
# 13.5 GB still leaves 2.5 GB headroom under 16 GB physical RAM, which
# keeps swap risk at zero. The 12 GB threshold was sized to catch
# new-allocation OOM cases like BGE-large; for reuse-only sweeps where
# model_est=0, the bloat-driven approach to 12 GB carries no swap risk
# and only blocks legitimate work. The cell-5 ram_preflight() applies
# this looser threshold ONLY when model_size_mb_estimate == 0; an explicit
# assert guard rejects the loosening on any fresh-load operation, so
# Exp1/Exp2's stricter 12 GB threshold remains intact for any future
# fresh-model-load experiments.
RAM_HALT_THRESHOLD_REUSE_ONLY_MB: int = 13_500  # 13.5 GB, reuse-only

# Prompt-length safety thresholds. Llama 3.1 8B modelfile default is
# num_ctx=131072 (128K); these thresholds flag prompts approaching that
# ceiling, but realistic Exp3 prompts top out at ~5300 tokens for top_k=10.
PROMPT_TOKEN_WARN_THRESHOLD: int = 4000
PROMPT_TOKEN_HARD_WARN_THRESHOLD: int = 7500


In [ ]:
"""Cell 3 — Imports, paths, determinism guards, eval questions.

One-time setup. Same scaffolding as 03 / 04, plus list-level validation of
TOP_K_VALUES_TO_RUN (positive ints, no duplicates).
"""
from __future__ import annotations

import random
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

_HERE = Path.cwd()
ROOT = _HERE if (_HERE / "MASTER_PLAN.md").exists() else _HERE.parent
assert (ROOT / "MASTER_PLAN.md").exists(), f"Could not find project root from {_HERE}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.eval_questions import load_eval_questions
from src.indexing import load_index
from src.metrics import (
    EXPECTED_INDEXING_COLS,
    EXPECTED_QUERY_COLS,
    NOTES_TAG_PROP_ENERGY,
    allocate_block_energy_proportionally,
    assert_indexing_schema,
    assert_query_sanity,
    assert_query_schema,
    count_retrieved_tokens,
    count_text_tokens,
    make_run_id,
    set_global_seeds,
)
from src.pipeline import RAGConfig, run_rag

# §9 C.5 — determinism guards.
random.seed(42)
np.random.seed(42)
set_global_seeds(42)

# Validate the driver tuple (cell 2).
assert isinstance(TOP_K_VALUES_TO_RUN, tuple), (
    "TOP_K_VALUES_TO_RUN must be a tuple of ints (cell 2)"
)
assert len(TOP_K_VALUES_TO_RUN) >= 1, "TOP_K_VALUES_TO_RUN is empty"
for _k in TOP_K_VALUES_TO_RUN:
    assert isinstance(_k, int) and _k >= 1, (
        f"TOP_K_VALUES_TO_RUN entries must be positive ints; got {_k!r}"
    )
assert len(set(TOP_K_VALUES_TO_RUN)) == len(TOP_K_VALUES_TO_RUN), (
    f"duplicate values in TOP_K_VALUES_TO_RUN={TOP_K_VALUES_TO_RUN}"
)

# Project paths.
RESULTS_DIR  = ROOT / "results"
CARBON_DIR   = RESULTS_DIR / "carbon"
INDICES_DIR  = ROOT / "indices"
EVAL_QS_PATH = ROOT / "dataset" / "eval_questions.md"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CARBON_DIR.mkdir(parents=True, exist_ok=True)

eval_questions = load_eval_questions(EVAL_QS_PATH)
type_counts: dict[str, int] = {}
for q in eval_questions:
    type_counts[q.question_type] = type_counts.get(q.question_type, 0) + 1

vm0 = psutil.virtual_memory()
print(f"[D.3] Project root:          {ROOT}")
print(f"[D.3] Frozen baseline:       chunk_size={FROZEN_CHUNK_SIZE}, "
      f"overlap_pct={FROZEN_OVERLAP_PCT}%, embedding={FROZEN_EMBEDDING}")
print(f"[D.3] EXP2_WINNER_HASH:      {EXP2_WINNER_HASH}")
print(f"[D.3] top_k sweep:           {TOP_K_VALUES_TO_RUN}")
print(f"[D.3] RAM halt threshold:    {RAM_HALT_THRESHOLD_MB} MB")
print(f"[D.3] Eval questions:        {len(eval_questions)} ; by type: {type_counts}")
print(f"[D.3] Memory at notebook startup: total={vm0.total/1024**2:.0f} MB, "
      f"available={vm0.available/1024**2:.0f} MB, "
      f"used={(vm0.total-vm0.available)/1024**2:.0f} MB ({vm0.percent:.1f}%)")


In [ ]:
"""Cell 4 — Locate the phase_d_exp1 source row, verify the artifact, then
load the index ONCE for the whole notebook.

The Exp3 indexing row is constructed by copying measurement fields from the
phase_d_exp1 row for aadb0cb9 (the ORIGINAL Exp1 measurement, not the
rebuilt phase_d_exp2 row). Both rows describe bit-identical artifacts, but
the user's pre-locked Exp3 plan specifies phase_d_exp1 as the source.
"""
EXP3_INDEX_DIR = INDICES_DIR / EXP2_WINNER_HASH

# 1) Verify the on-disk artifact.
artifact_ok = (
    (EXP3_INDEX_DIR / "faiss.index").exists()
    and (EXP3_INDEX_DIR / "chunks.parquet").exists()
)
assert artifact_ok, (
    f"Index artifact missing at {EXP3_INDEX_DIR}/. Expected faiss.index + "
    f"chunks.parquet from Phase D Exp1. Re-run Exp1 to regenerate."
)

# 2) Locate the phase_d_exp1 source row in indexing_log.csv.
idx_log_path = RESULTS_DIR / "indexing_log.csv"
assert idx_log_path.exists(), f"indexing_log.csv missing at {idx_log_path}"
existing_idx = pd.read_csv(idx_log_path)
exp1_match = existing_idx[
    (existing_idx["config_hash"] == EXP2_WINNER_HASH)
    & (existing_idx["notes"] == "phase_d_exp1")
]
assert len(exp1_match) == 1, (
    f"Expected exactly 1 phase_d_exp1 row for hash={EXP2_WINNER_HASH}, "
    f"got {len(exp1_match)}. Inspect results/indexing_log.csv."
)
PHASE_D_EXP1_SOURCE_ROW = exp1_match.iloc[0]
print(f"[D.3] phase_d_exp1 source row located:")
print(f"      run_id={PHASE_D_EXP1_SOURCE_ROW['run_id']}")
print(f"      indexing_time_sec={PHASE_D_EXP1_SOURCE_ROW['indexing_time_sec']:.2f}, "
      f"peak_ram_mb={PHASE_D_EXP1_SOURCE_ROW['peak_ram_mb']:.1f}, "
      f"energy_wh={PHASE_D_EXP1_SOURCE_ROW['energy_wh']:.4f}")
print(f"      n_chunks_total={PHASE_D_EXP1_SOURCE_ROW['n_chunks_total']}, "
      f"index_size_mb={PHASE_D_EXP1_SOURCE_ROW['index_size_mb']:.2f}")

# 3) Load the index ONCE. The (faiss_index, chunks) pair is reused across
#    all 4 top_k configs × 3 reps × 13 questions in cell 6.
print(f"[D.3] Loading index from {EXP3_INDEX_DIR} (once for the whole notebook)...")
t0 = time.perf_counter()
faiss_index, chunks = load_index(EXP3_INDEX_DIR)
load_dt = time.perf_counter() - t0
print(f"[D.3] Loaded {faiss_index.ntotal} vectors / {len(chunks)} chunks "
      f"in {load_dt*1000:.1f} ms.")

# 4) Sanity: top_k=10 must not exceed n_chunks (would FAISS-search-limit).
max_k = max(TOP_K_VALUES_TO_RUN)
assert faiss_index.ntotal >= max_k, (
    f"max(TOP_K_VALUES_TO_RUN)={max_k} > index.ntotal={faiss_index.ntotal} "
    f"— retrieval would return fewer than k chunks for some questions."
)
print(f"[D.3] max(top_k)={max_k} <= index.ntotal={faiss_index.ntotal}: OK")


In [ ]:
"""Cell 5 — Helpers.

ram_preflight(): defensive check before each top_k's query block.
Threshold same as Exp2 (12 GB). Low risk this experiment — no new model
loads — but kept as a safety rail against background-process drift.

run_one_topk(): per-top_k end-to-end. For one k value:
  - Builds RAGConfig with that k (config_hash is unchanged: aadb0cb9).
  - For r ∈ {1, 2, 3}: opens a fresh OfflineEmissionsTracker, runs all 13
    questions against the SHARED (faiss_index, chunks), allocates block
    energy/CO2 proportionally by total_latency_ms, appends 13 query rows
    with run_id = f"{idx_run_id}_k{k}_r{r}".
  - After the 3 reps, runs schema lock-in + sanity asserts on rows
    filtered by (run_id.startswith(idx_run_id)) & (top_k == k). Filters
    by top_k explicitly because all 4 Exp3 configs share idx_run_id; the
    top_k column is what differentiates them.
"""
from codecarbon import OfflineEmissionsTracker

REFUSAL_PHRASE = "I don't have enough information"
EMBEDDING_SPIKE_BUFFER_MB = 500.0  # same convention as Exp2


def ram_preflight(
    model_size_mb_estimate: float = 0.0,
    *,
    halt_threshold_mb: int = RAM_HALT_THRESHOLD_MB,
) -> tuple[bool, str]:
    """Project resident-memory usage and decide whether to halt.

    Applies the loosened RAM_HALT_THRESHOLD_REUSE_ONLY_MB (13.5 GB) ONLY
    when ``model_size_mb_estimate == 0`` — i.e. the caller is running a
    reuse-only sweep with no new model loads (Exp3 pattern). For any
    fresh-load operation (model_est > 0), the strict 12 GB default
    applies. An explicit assert guard rejects the loosening on any
    fresh-load operation; see §13.1 in MASTER_PLAN.md for the rationale.
    """
    # Threshold selection with explicit guard (§13.1).
    if model_size_mb_estimate == 0.0:
        # Reuse-only sweep: no new allocations, only heap bloat from
        # accumulated objects. The looser threshold avoids halting on
        # bloat that carries no swap risk under 16 GB physical RAM.
        effective_threshold_mb = RAM_HALT_THRESHOLD_REUSE_ONLY_MB
    else:
        # Fresh-model-load operation: keep the strict default that
        # catches genuine OOM risk (Exp1/Exp2 pattern).
        effective_threshold_mb = halt_threshold_mb

    # Guard: the looser threshold MUST NEVER apply with model_est > 0.
    if effective_threshold_mb > RAM_HALT_THRESHOLD_MB:
        assert model_size_mb_estimate == 0.0, (
            f"GUARD VIOLATION: looser threshold {effective_threshold_mb} MB "
            f"can only be applied when model_size_mb_estimate == 0; got "
            f"{model_size_mb_estimate}. See §13.1 in MASTER_PLAN.md "
            f"(single-machine policy)."
        )

    vm = psutil.virtual_memory()
    used_mb_now = (vm.total - vm.available) / 1024**2
    proj_used_mb = used_mb_now + model_size_mb_estimate + EMBEDDING_SPIKE_BUFFER_MB
    proj_avail_mb = (vm.total / 1024**2) - proj_used_mb
    threshold_tag = (
        " [reuse-only loosening, §13.1]"
        if effective_threshold_mb > RAM_HALT_THRESHOLD_MB
        else ""
    )
    msg = (
        f"used_now={used_mb_now:.0f} MB, model_est={model_size_mb_estimate:.0f}, "
        f"spike_buffer={EMBEDDING_SPIKE_BUFFER_MB:.0f} -> "
        f"proj_used={proj_used_mb:.0f} MB "
        f"(threshold {effective_threshold_mb} MB{threshold_tag}), "
        f"proj_available={proj_avail_mb:.0f} MB"
    )
    return proj_used_mb < effective_threshold_mb, msg


def run_one_topk(
    k: int,
    *,
    exp3_ts_iso: str,
    idx_run_id: str,
    faiss_index,
    chunks,
) -> dict:
    cfg = RAGConfig(
        chunk_size=FROZEN_CHUNK_SIZE,
        overlap_pct=FROZEN_OVERLAP_PCT,
        embedding_model=FROZEN_EMBEDDING,
        top_k=k,
    )
    # All 4 Exp3 configs share this hash (top_k is not a hash input).
    assert cfg.hash == EXP2_WINNER_HASH, (
        f"BUG: Exp3 cfg.hash={cfg.hash!r} != EXP2_WINNER_HASH="
        f"{EXP2_WINNER_HASH!r}"
    )

    print()
    print("=" * 78)
    print(f"[D.3] top_k={k}  (hash={cfg.hash}, shared across all 4 Exp3 configs)")
    print("=" * 78)

    # Pre-query-loop RAM pre-flight (defensive; no model load expected).
    ok, ram_msg = ram_preflight(0.0)
    print(f"[D.3] RAM pre-flight: {ram_msg}")
    if not ok:
        raise RuntimeError(
            f"RAM pre-flight HALT before top_k={k} query loop: {ram_msg}"
        )

    rep_summaries: list[dict] = []
    for r in (1, 2, 3):
        # Embed top_k in qry_run_id so all 12 (k, r) combinations get
        # distinct run_ids while still prefix-joining to idx_run_id.
        qry_run_id = f"{idx_run_id}_k{k}_r{r}"
        assert qry_run_id.startswith(idx_run_id), (
            f"BUG: qry_run_id={qry_run_id!r} not prefixed by "
            f"idx_run_id={idx_run_id!r}"
        )

        qry_tracker = OfflineEmissionsTracker(
            project_name=f"phase_d_exp3_query_{cfg.hash}_k{k}_r{r}",
            tracking_mode="machine",
            country_iso_code="DEU",
            output_dir=str(CARBON_DIR),
            output_file="emissions.csv",
            measure_power_secs=1.0,
            log_level="error",
            save_to_file=True,
        )

        rows: list[dict] = []
        rep_t0 = time.perf_counter()
        prompt_token_max = 0
        loud_warns = 0
        qry_tracker.start()
        try:
            for q in eval_questions:
                result = run_rag(q.question, cfg, faiss_index, chunks)
                t = result["timings"]
                retrieved = result["retrieved_chunks"]

                embed_ms    = t["embed"]    * 1000.0
                retrieve_ms = t["retrieve"] * 1000.0
                generate_ms = t["generate"] * 1000.0
                total_ms    = embed_ms + retrieve_ms + generate_ms

                ptc = int(count_text_tokens(result["prompt"]))
                if ptc > prompt_token_max:
                    prompt_token_max = ptc
                if ptc > PROMPT_TOKEN_HARD_WARN_THRESHOLD:
                    loud_warns += 1
                    print(f"[D.3]   WARN: {q.question_id} prompt_token_count="
                          f"{ptc} exceeds {PROMPT_TOKEN_HARD_WARN_THRESHOLD} "
                          f"— Llama may truncate context.")

                rows.append({
                    "run_id":                qry_run_id,
                    "timestamp":             exp3_ts_iso,
                    "config_hash":           cfg.hash,
                    "question_id":           q.question_id,
                    "top_k":                 int(cfg.top_k),
                    "query_embed_time_ms":   float(embed_ms),
                    "retrieval_time_ms":     float(retrieve_ms),
                    "generation_time_ms":    float(generate_ms),
                    "total_latency_ms":      float(total_ms),
                    "n_retrieved_chunks":    int(len(retrieved)),
                    "retrieved_token_count": int(count_retrieved_tokens(retrieved)),
                    "prompt_token_count":    ptc,
                    "answer_token_count":    int(count_text_tokens(result["answer"])),
                    "energy_wh_per_query":   None,
                    "co2_g_per_query":       None,
                    "retrieved_chunk_ids":   ";".join(rr.chunk.chunk_id for rr in retrieved),
                    "answer_text":           result["answer"],
                    "notes":                 "",
                })
        finally:
            qry_co2_kg = qry_tracker.stop()
        rep_wall_s = time.perf_counter() - rep_t0

        qry_energy_wh = qry_tracker.final_emissions_data.energy_consumed * 1000.0
        qry_co2_g     = qry_co2_kg * 1000.0

        allocate_block_energy_proportionally(
            rows, qry_energy_wh, qry_co2_g,
        )

        qry_csv = RESULTS_DIR / "query_log.csv"
        write_header = not qry_csv.exists()
        pd.DataFrame(rows, columns=EXPECTED_QUERY_COLS).to_csv(
            qry_csv, mode="a", index=False, header=write_header,
        )

        n_refusals = sum(
            1 for row in rows if REFUSAL_PHRASE in row["answer_text"]
        )
        mean_total_ms = sum(row["total_latency_ms"] for row in rows) / len(rows)
        rep_summaries.append({
            "rep": r,
            "qry_run_id": qry_run_id,
            "energy_wh": qry_energy_wh,
            "co2_g": qry_co2_g,
            "n_refusals": n_refusals,
            "mean_total_ms": mean_total_ms,
            "rep_wall_s": rep_wall_s,
            "prompt_token_max": prompt_token_max,
            "loud_warns": loud_warns,
        })
        print(f"[D.3]   k={k} rep {r}: wall={rep_wall_s:.1f}s "
              f"mean_total_latency={mean_total_ms/1000:.1f}s "
              f"refusals={n_refusals}/13 "
              f"qry_energy={qry_energy_wh:.4f} Wh  "
              f"prompt_max={prompt_token_max}  qry_run_id={qry_run_id}")

    # Per-config sanity asserts. Filter by run_id prefix AND top_k column —
    # the indexing run_id is shared across all 4 Exp3 configs, so the
    # top_k filter is required to isolate the 39 rows for THIS k.
    idx_df_full = pd.read_csv(RESULTS_DIR / "indexing_log.csv")
    qry_df_full = pd.read_csv(RESULTS_DIR / "query_log.csv")
    assert_indexing_schema(idx_df_full)
    assert_query_schema(qry_df_full)

    idx_this = idx_df_full[idx_df_full["run_id"] == idx_run_id]
    qry_this = qry_df_full[
        (qry_df_full["run_id"].astype(str).str.startswith(idx_run_id))
        & (qry_df_full["top_k"] == k)
    ]
    assert_query_sanity(
        qry_this, idx_this,
        top_k=k,
        chunk_size=FROZEN_CHUNK_SIZE,
        expected_query_count=39,
        expected_indexing_count=1,
        notes_tag=NOTES_TAG_PROP_ENERGY,
    )
    print(f"[D.3] Sanity asserts (schema lock + A–H) PASSED for "
          f"top_k={k} (39 query rows; shared idx_run_id={idx_run_id})")

    return {
        "top_k":           k,
        "config_hash":     cfg.hash,
        "rep_summaries":   rep_summaries,
    }


In [ ]:
"""Cell 6 — Emit the SHARED indexing row once, then sweep top_k values."""

# 1) Anchor a single UTC timestamp for the whole Exp3 sweep. This anchors
#    both the (single) idx_run_id and all 12 query run_ids, so the
#    indexing run_id is a strict prefix of every query run_id.
EXP3_TS = datetime.now(timezone.utc)
exp3_ts_iso, exp3_ts_compact, EXP3_IDX_RUN_ID = make_run_id(
    EXP2_WINNER_HASH, ts=EXP3_TS,
)
print()
print("=" * 78)
print(f"[D.3] Exp3 anchor: {exp3_ts_iso}")
print(f"[D.3] EXP3_IDX_RUN_ID = {EXP3_IDX_RUN_ID}")
print("=" * 78)

# 2) Build the indexing row by copying measurement fields from the
#    phase_d_exp1 source row identified in cell 4. Override: run_id,
#    timestamp, notes (and embedding_model, which is already correct
#    since we filtered the source row by config_hash anyway).
src = PHASE_D_EXP1_SOURCE_ROW  # 1-row Series from cell 4
indexing_row = {
    "run_id":               EXP3_IDX_RUN_ID,
    "timestamp":            exp3_ts_iso,
    "config_hash":          str(src["config_hash"]),
    "chunk_size":           int(src["chunk_size"]),
    "chunk_overlap":        int(src["chunk_overlap"]),
    "embedding_model":      str(src["embedding_model"]),
    "n_documents":          int(src["n_documents"]),
    "n_chunks_total":       int(src["n_chunks_total"]),
    "indexing_time_sec":    float(src["indexing_time_sec"]),
    "peak_ram_mb":          float(src["peak_ram_mb"]),
    "index_size_mb":        float(src["index_size_mb"]),
    "embedding_time_sec":   float(src["embedding_time_sec"]),
    "faiss_build_time_sec": float(src["faiss_build_time_sec"]),
    "energy_wh":            float(src["energy_wh"]),
    "co2_g":                float(src["co2_g"]),
    "notes":                "phase_d_exp3 (reused phase_d_exp1 "
                            f"{EXP2_WINNER_HASH} indexing measurements)",
}

idx_csv = RESULTS_DIR / "indexing_log.csv"
write_header = not idx_csv.exists()
pd.DataFrame([indexing_row], columns=EXPECTED_INDEXING_COLS).to_csv(
    idx_csv, mode="a", index=False, header=write_header,
)
print(f"[D.3] Wrote 1 indexing row (notes='{indexing_row['notes']}').")

# 3) Top_k sweep. faiss_index and chunks come from cell 4's load_index.
all_topk_results: list[dict] = []
total_t0 = time.perf_counter()
for k in TOP_K_VALUES_TO_RUN:
    summary = run_one_topk(
        k,
        exp3_ts_iso=exp3_ts_iso,
        idx_run_id=EXP3_IDX_RUN_ID,
        faiss_index=faiss_index,
        chunks=chunks,
    )
    all_topk_results.append(summary)
total_wall_s = time.perf_counter() - total_t0

print()
print("#" * 78)
print(f"# Exp3 Run All complete: total wall = {total_wall_s/60:.1f} min "
      f"({total_wall_s:.0f} s) across {len(all_topk_results)} top_k values")
print("#" * 78)


In [ ]:
"""Cell 7 — Combined Exp3 summary + winner computation.

Section 1 — per-top_k detail for THIS Run All (in-memory).
Section 2 — winner ranking from CSV (authoritative).

Winner rule (mirrors Exp1 / Exp2, locked):
  primary       refusal_rate          ASC
  tie-break #1  mean_total_latency_ms ASC
  tie-break #2  indexing_time_sec     ASC

NOTE: indexing_time_sec is identical across all 4 Exp3 configs (one
shared indexing row), so tie-break #2 is mathematically incapable of
resolving any tie. If a tie persists after tie-break #1, cell 7 prints
all tied configs and halts winner selection — the user must adjudicate.
"""

# ----- Section 1 -----------------------------------------------------------
print()
print("=" * 78)
print(f"Phase D / Exp3 — per-top_k summary for THIS Run All  "
      f"({len(all_topk_results)} configs)")
print("=" * 78)

for s in all_topk_results:
    n_ref_per_rep    = [rep["n_refusals"]    for rep in s["rep_summaries"]]
    mean_lat_per_rep = [rep["mean_total_ms"] for rep in s["rep_summaries"]]
    energy_per_rep   = [rep["energy_wh"]     for rep in s["rep_summaries"]]
    prompt_max_rep   = [rep["prompt_token_max"] for rep in s["rep_summaries"]]
    mean_refusal_rate     = sum(n_ref_per_rep) / (3 * 13)
    mean_total_latency_ms = sum(mean_lat_per_rep) / 3.0
    print()
    print(f"  config: top_k={s['top_k']:>2}  hash={s['config_hash']}")
    print(f"    refusals/rep   : {n_ref_per_rep}  "
          f"-> mean_refusal_rate={mean_refusal_rate:.1%}")
    print(f"    mean_total_ms/rep: {[f'{x:.0f}' for x in mean_lat_per_rep]}  "
          f"-> mean={mean_total_latency_ms:.0f}")
    print(f"    qry_energy_wh/rep: {[f'{x:.4f}' for x in energy_per_rep]}")
    print(f"    prompt_token_max/rep: {prompt_max_rep}")

# ----- Section 2 -----------------------------------------------------------
print()
print("=" * 78)
print("Phase D / Exp3 — CROSS-RUN Pareto ranking "
      "(loads ALL phase_d_exp3 rows from CSV)")
print("=" * 78)

idx_full = pd.read_csv(RESULTS_DIR / "indexing_log.csv")
qry_full = pd.read_csv(RESULTS_DIR / "query_log.csv")

# All phase_d_exp3 indexing rows — covers both partial-run rows
# (e.g. the prior top_k=1 run) AND this run's row.
exp3_idx = idx_full[
    idx_full["notes"].astype(str).str.startswith("phase_d_exp3")
].copy()
print(f"Found {len(exp3_idx)} phase_d_exp3 indexing rows in CSV "
      f"(expect 2: 1 prior partial top_k=1 run + 1 this run).")
for _, irow in exp3_idx.iterrows():
    print(f"  run_id={irow['run_id']}  config_hash={irow['config_hash']}")
assert len(exp3_idx) >= 1, "FAIL: no phase_d_exp3 indexing row found"

# Sanity: indexing_time_sec must be identical across all phase_d_exp3
# rows (every row copies its measurements from the same phase_d_exp1
# source row).
unique_idx_times = exp3_idx["indexing_time_sec"].unique()
assert len(unique_idx_times) == 1, (
    f"FAIL: phase_d_exp3 rows disagree on indexing_time_sec: "
    f"{unique_idx_times}. Expected all-equal (copied from one phase_d_exp1 row)."
)
exp3_indexing_time_sec = float(unique_idx_times[0])

# Collect every query row whose run_id starts with ANY phase_d_exp3
# idx_run_id. This catches all reps of all top_k configs across all
# Exp3 Run-Alls.
exp3_idx_run_ids = exp3_idx["run_id"].astype(str).tolist()
exp3_qry_mask = qry_full["run_id"].astype(str).apply(
    lambda rid: any(rid.startswith(idx_rid) for idx_rid in exp3_idx_run_ids)
)
exp3_qry = qry_full[exp3_qry_mask].copy()
expected_total = len(TOP_K_VALUES_FOR_EXP3_RANKING) * 3 * 13
print(f"Query rows under any phase_d_exp3 idx_run_id: {len(exp3_qry)} "
      f"(expect {expected_total} = "
      f"{len(TOP_K_VALUES_FOR_EXP3_RANKING)} top_k × 3 reps × 13 q)")

# Aggregate by top_k across the FULL Exp3 set.
ranked_rows: list[dict] = []
for k in TOP_K_VALUES_FOR_EXP3_RANKING:
    qsel = exp3_qry[exp3_qry["top_k"] == k]
    if len(qsel) != 39:
        print(f"  WARNING: top_k={k} has {len(qsel)} query rows, "
              f"expected 39 — excluding from ranking.")
        continue
    n_ref = qsel["answer_text"].astype(str).str.contains(
        REFUSAL_PHRASE, regex=False,
    ).sum()
    refusal_rate = n_ref / 39.0
    mean_total_latency_ms = float(qsel["total_latency_ms"].mean())
    ranked_rows.append({
        "top_k":                 int(k),
        "refusal_rate":          float(refusal_rate),
        "n_refusals":            int(n_ref),
        "mean_total_latency_ms": mean_total_latency_ms,
        "indexing_time_sec":     exp3_indexing_time_sec,
    })

ranked = pd.DataFrame(ranked_rows).sort_values(
    ["refusal_rate", "mean_total_latency_ms", "indexing_time_sec"],
    kind="mergesort",
).reset_index(drop=True)

print()
header = (f"{'rank':>4}  {'top_k':>5}  {'refusal':>8}  {'n_ref/39':>8}  "
          f"{'mean_lat_ms':>11}  {'idx_time_s':>10}")
print(header)
print("-" * len(header))
for r, row in ranked.iterrows():
    print(f"{r+1:>4}  {row.top_k:>5}  {row.refusal_rate:>7.1%}  "
          f"{row.n_refusals:>8}  {row.mean_total_latency_ms:>11.0f}  "
          f"{row.indexing_time_sec:>10.2f}")

# Tie detection: if 2+ configs share refusal_rate AND mean_total_latency_ms,
# tie-break #2 (indexing_time_sec) cannot help (constant for Exp3). Halt
# selection and let the user adjudicate.
top_refusal = ranked.iloc[0].refusal_rate
top_lat     = ranked.iloc[0].mean_total_latency_ms
ties = ranked[
    (ranked["refusal_rate"] == top_refusal)
    & (ranked["mean_total_latency_ms"] == top_lat)
]

print()
print("=" * 78)
if len(ties) > 1:
    print(f"EXP3 WINNER  AMBIGUOUS — {len(ties)} configs tied on both primary "
          f"and tie-break #1; tie-break #2 (indexing_time_sec) is constant "
          f"this experiment and cannot resolve. User adjudication required.")
    print("=" * 78)
    print("Tied configs:")
    for _, t in ties.iterrows():
        print(f"  top_k={int(t.top_k):>2}  refusal_rate={t.refusal_rate:.1%}  "
              f"mean_total_latency_ms={t.mean_total_latency_ms:.0f}  "
              f"n_refusals={int(t.n_refusals)}")
else:
    winner = ranked.iloc[0]
    print("EXP3 WINNER")
    print("=" * 78)
    print(f"  top_k:                 {int(winner.top_k)}")
    print(f"  config_hash:           {EXP2_WINNER_HASH}  (shared across all Exp3 configs)")
    print(f"  mean refusal rate:     {winner.refusal_rate:.1%}  "
          f"({int(winner.n_refusals)}/39)")
    print(f"  mean total_latency_ms: {winner.mean_total_latency_ms:.0f}")
    print(f"  indexing_time_sec:     {winner.indexing_time_sec:.2f}  "
          f"(shared across all Exp3 configs — not a discriminator)")


## Wrap-up — what this notebook produced

For top_k ∈ {1, 3, 5, 10} swept against the frozen Exp2 winner
(`config_hash=aadb0cb9`, MiniLM-L6-v2 at chunk_size=512 / overlap_pct=20%):

- **1 row** appended to `results/indexing_log.csv` (notes:
  `phase_d_exp3 (reused phase_d_exp1 aadb0cb9 indexing measurements)`).
- **156 rows** appended to `results/query_log.csv` (4 top_k × 3 reps × 13
  questions, each tagged with `NOTES_TAG_PROP_ENERGY` per Phase C Q2).
- **12 query emission blocks** in `results/carbon/emissions.csv`
  (`project_name=phase_d_exp3_query_<hash>_k<k>_r{r}`). **Zero indexing
  emission blocks** (artifact reuse — no real work to measure).
- **Per-top_k sanity asserts** — schema lock-in on both CSVs, plus A–H
  value asserts (`src.metrics.assert_query_sanity`) on rows filtered by
  `(run_id.startswith(idx_run_id)) & (top_k == k)`.

Phase D Exp3 closes Phase D entirely (D.1 ✅, D.2 ✅, D.3 ✅). The Phase D
status row in MASTER_PLAN.md §4 should be flipped 🟡 → ✅ in the
PROGRESS_LOG entry that closes this notebook's session, alongside a
consolidated cumulative-state tally:
  - 13 indexing rows total: 1 Phase C + 9 Phase D Exp1 + 2 Phase D Exp2
    + 1 Phase D Exp3.
  - Query rows: 13 Phase C + 351 Phase D Exp1 + 78 Phase D Exp2 + 156
    Phase D Exp3 = 598 total. (The §10 D.4 plan-line of 624 query rows
    referenced 16 configs × 13 q × 3 reps including BGE-large; the
    delta of 39 rows reflects the Exp2 BGE drop documented under §14.)
